# Modélisation & Évaluation — Adult Income (UCI)

Ce notebook utilise les modules `ModelRegistry` (catalogue de classifieurs) et `ModelEvaluator` (cross-validation, évaluation test, comparaisons & graphes).

Pipeline : `DataLoader` → `PreprocessingPipeline` → modèles du `ModelRegistry` → `ModelEvaluator`.

In [ ]:
# Recharge automatique des modules src/ modifiés sans restart du kernel
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Permet d'importer le package src/ depuis le dossier notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import matplotlib.pyplot as plt

from adult_income import (
    DataLoader,
    ModelEvaluator,
    ModelRegistry,
    PreprocessingPipeline,
)

print("Imports OK")

## 1 — Données & preprocessing

In [ ]:
# --- Données : split officiel (adult.data / adult.test) ---
loader = DataLoader()
X_train, X_test, y_train, y_test = loader.load_and_split()

# --- Preprocessing (one-hot + scaling, adapté aux modèles linéaires) ---
pp = PreprocessingPipeline(encoding="onehot", scaling=True)

# Encoder la cible (<=50K -> 0, >50K -> 1)
y_train_enc = pp.fit_transform_target(y_train)
y_test_enc = pp.transform_target(y_test)

print(f"Train : {X_train.shape[0]:,} obs | Test : {X_test.shape[0]:,} obs")

## 2 — Catalogue de modèles (`ModelRegistry`)

In [ ]:
# Tous les modèles disponibles dans le catalogue
print("Modèles disponibles :")
for name in ModelRegistry.available():
    print(f"  - {name}")

In [ ]:
# Sous-ensemble à comparer (élargis la liste avec ModelRegistry.available())
model_names = ModelRegistry.available()  # ['logistic_regression', 'random_forest', 'gradient_boosting', 'xgboost', 'lightgbm']

models = ModelRegistry.get_subset(model_names)

# Pipelines complets : preprocessing + classifieur
pipelines = {name: pp.build_pipeline(est) for name, est in models.items()}
pipelines

## 3 — Cross-validation sur le train set (`ModelEvaluator`)

In [ ]:
evaluator = ModelEvaluator(cv=5, random_state=42)
evaluator.cross_validate_all(pipelines, X_train, y_train_enc)

evaluator.cv_comparison_table()

In [ ]:
evaluator.plot_cv_comparison(metric="roc_auc")
plt.show()

## 4 — Évaluation finale sur le test set

In [ ]:
for name, pipeline in pipelines.items():
    evaluator.evaluate_on_test(name, pipeline, X_train, y_train_enc, X_test, y_test_enc)

evaluator.test_comparison_table()

In [ ]:
evaluator.plot_metric_heatmap()
plt.show()

In [ ]:
evaluator.plot_roc_curves(X_test, y_test_enc)
plt.show()

## 5 — Focus sur le meilleur modèle

In [ ]:
best = evaluator.test_comparison_table().iloc[0]["model"]
print(f"Meilleur modèle (roc_auc test) : {best}")

evaluator.print_classification_report(best, pipelines[best], X_test, y_test_enc)

In [ ]:
evaluator.plot_confusion_matrix(best, X_test, y_test_enc)
plt.show()